# 02 — Phase 2 (T_t transfer control) + Phase 2b (activation Q metrics)

Phase 2 is cheap and MUST be reported to the channel before Phase 3 (notebook 03) starts — it decides how `ΔR_t` can be read (plan §2 reading rule). Phase 2b gates nothing downstream; if time is short, it is the first thing to cut (plan §4 de-scope priority order item 4).

In [ ]:
%pip install -q -r "/content/RLVR/experiment 2/requirements.txt"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
REPO_URL = 'https://github.com/WYR186/RLVR.git'  # HTTPS; if the repo is
# private, authenticate interactively (git credential prompt / a token you
# paste when asked) rather than embedding a token in this notebook.
REPO_DIR = '/content/RLVR'
import os
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

EXP2_DIR = f'{REPO_DIR}/experiment 2'
sys.path.insert(0, EXP2_DIR)  # only this one goes on sys.path — pipeline.py
# reaches eaaj-pilot/src by explicit file path internally, avoiding a
# top-level `src` package-name collision between the two sibling dirs
# (see experiment 2/src/pipeline.py's module docstring).

import src.guru_data as guru_data
import src.guru_reward as guru_reward
import src.pipeline as pipeline

import json
from pathlib import Path
CONFIG = json.load(open(f'{EXP2_DIR}/exp2_colab_config.json'))
DATA_DIR = Path(EXP2_DIR) / 'data'
print('config loaded:', CONFIG['experiment'])

In [ ]:
splits = json.loads((DATA_DIR / 'exp2_splits.json').read_text())
domain_field = splits['domain_field']
PROMPT_FIELD, ANSWER_FIELD = splits['prompt_field'], splits['answer_field']
raw = guru_data._load_raw(revision=splits['dataset_revision'])
train_split = raw['train'] if 'train' in raw else next(iter(raw.values()))
stage_b_pool = guru_data.filter_stage_subset(train_split, guru_data.STAGE_B_SUBSET_NAMES, domain_field)

stage_b_eval = stage_b_pool.select(splits['stage_b_eval_idx'])
eval_prompts = [str(r[PROMPT_FIELD]) for r in stage_b_eval]
eval_golds = [str(r[ANSWER_FIELD]) for r in stage_b_eval]
print('stage-B eval set:', len(eval_prompts))

RUN_DIR = f'{EXP2_DIR}/../eaaj-pilot/outputs/exp2_colab_guru_math7b_REPLACE_WITH_HASH'
STAGE_A_DIR = f'{RUN_DIR}/stage_a'
sa = CONFIG['stage_a']

## Phase 2 — T_t (zero-shot, no stage-2 training)

In [ ]:
transfer = pipeline.run_transfer_T(
    CONFIG['model_id'], CONFIG['peft'], STAGE_A_DIR, sa['checkpoint_steps'],
    eval_prompts, eval_golds, f'{RUN_DIR}/analysis/transfer_T.json')
print(transfer)
print('Report T_t to the channel before starting notebook 03.')

## Phase 2b — activation Q metrics

Probe prompts come from the **frozen probe split** in `exp2_splits.json` (disjoint from stage-B train AND eval; topped up from held-out stage-A rows if the CodeI/O pool was too small — the per-source counts are in the splits file). Never substitute the eval set here: it is only 300 prompts and the effective-rank magnitudes need n_probe > hidden dim 3584 (plan §1).

In [ ]:
m = CONFIG['measurement']
probe_b = stage_b_pool.select(splits['probe_stage_b_idx'])
probe_prompts = [str(r[PROMPT_FIELD]) for r in probe_b]
if splits['probe_stage_a_topup_idx']:
    stage_a_pool = guru_data.filter_stage_subset(train_split, guru_data.STAGE_A_SUBSET_NAMES, domain_field)
    probe_a = stage_a_pool.select(splits['probe_stage_a_topup_idx'])
    probe_prompts += [str(r[PROMPT_FIELD]) for r in probe_a]
print('probe size:', len(probe_prompts), '(requested', m['probe_questions'], ')',
      '| stage-B rows:', len(splits['probe_stage_b_idx']),
      '| stage-A topup:', len(splits['probe_stage_a_topup_idx']))
if len(probe_prompts) < 3584:
    print('WARNING: probe smaller than hidden dim (3584) — erank magnitudes are '
          'sample-truncated; report n_probe alongside every value (plan §1).')

from pathlib import Path as _Path
_Path(f'{RUN_DIR}/measurements').mkdir(parents=True, exist_ok=True)
q_by_ckpt = {}
for step in sa['checkpoint_steps']:
    ckpt_dir = f'{STAGE_A_DIR}/ckpt-{step}'
    q = pipeline.measure_checkpoint_q(
        CONFIG['model_id'], CONFIG['peft'], ckpt_dir, probe_prompts,
        layers=tuple(m['layers']), batch_size=m['batch_size'])
    q_by_ckpt[step] = q
    json.dump(q, open(f'{RUN_DIR}/measurements/metrics_ckpt{step}.json', 'w'), indent=1, default=str)
    print('measured ckpt', step)

## ckpt-0 within-run identity re-check (plan §1 — not a cross-model comparison)

In [ ]:
q_ckpt0_again = pipeline.measure_checkpoint_q(
    CONFIG['model_id'], CONFIG['peft'], f'{STAGE_A_DIR}/ckpt-0', probe_prompts,
    layers=tuple(m['layers']), batch_size=m['batch_size'])
import math
l = m['layers'][0]
erank_1 = q_by_ckpt[0]['per_layer'][f'layer{l}']['erank']
erank_2 = q_ckpt0_again['per_layer'][f'layer{l}']['erank']
delta = abs(erank_1 - erank_2)
print(f'ckpt-0 identity re-check, layer {l}: erank {erank_1:.6f} vs {erank_2:.6f}, delta={delta:.2e}')
if delta > 1e-4:
    print('WARNING: measurement contract may have drifted between the two ckpt-0 passes — investigate before trusting Phase 2b.')

## Commit reminder

Commit `analysis/transfer_T.json` and `measurements/metrics_ckpt*.json`, prefix `exp2-colab:`.